In [1]:
#******************************************************
#*
#* Name:         nb-04-python-ingest-csv-files
#*     
#* Design Phase:
#*     Author:   John Miner
#*     Date:     04-01-2026
#*     Purpose:   Load delta table using python.
#* 
#******************************************************/

In [2]:
%%capture

#
#  Get lakehouse abfs path
#

def get_lakehouse_abfs_path(key = "/default"):
    
    # get mount point
    try:
        path = list(filter(lambda x: x["mountPoint"] == key, notebookutils.fs.mounts()))[0]["source"]
    except:
        path = ""

    # return results
    return path

# get the path
lakehouse_path = get_lakehouse_abfs_path()


In [3]:
#
#  1 - create function to read files recursively
#

import pandas as pd
from pathlib import Path

def read_csv_recursively(root_dir: str, pattern: str = "*.csv") -> pd.DataFrame:

    # instantiate object
    root_path = Path(root_dir)

    # get file list
    csv_files = root_path.rglob(pattern)

    # empty list
    frames = []

    # for each file, read and append to list
    for file in csv_files:
        df = pd.read_csv(file)  
        frames.append(df)

    # append each element
    if frames:
        return pd.concat(frames, ignore_index=True)

    # no file found case
    else:
        return pd.DataFrame() 


In [4]:
#
#  2 - execute the function (execution time = 1 min 50 s )
#

df = read_csv_recursively(
    root_dir='/lakehouse/default/Files/stocks',
    pattern="*.CSV"
)


/tmp/ipykernel_133/4242379903.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


In [5]:
#
#  3 - peak at the data
#

# add prefix
df1 = df.rename(columns={c: 'st_' + c for c in df.columns})

# get rid of dividends
df1 = df1[df1['st_volume'] != 0]

# count recs
df1.shape


(626714, 8)

In [6]:
#
# 4 - remove target table
#

# set variables
schema_name = "bronze"
table_name = "pandas_stock_data"
table_path = f"{lakehouse_path}/Tables/{schema_name}/{table_name}"

# only fails on missing dir
try:
    notebookutils.fs.rm(table_path, True)
except:
    pass

In [7]:
#
# 5 - write delta lake table (7 sec)
#

from deltalake import write_deltalake

# message user
print(f"create delta table located at {table_path}")

# over write 
write_deltalake(table_path, df1, mode='overwrite')


create delta table located at abfss://9d12d1b1-e32e-4bbe-9753-f0715d21ad49@onelake.dfs.fabric.microsoft.com/f880ba3e-79f9-4297-8b7f-4e39ca3d158b/Tables/bronze/pandas_stock_data


In [8]:

#
#  6 - Write single file (3 sec)
#

# write file
path = '/lakehouse/default/Files/SNP-5YEARS.CSV'
df1.to_csv(path, sep='\t', encoding='utf-8', index=False, header=True)


In [9]:
#
#  7 - execute the function (1 sec)
#

df2 = pd.read_csv("/lakehouse/default/Files/SNP-5YEARS.CSV", sep='\t')


In [10]:
#
# 8 - remove target table
#

# set variables
schema_name = "bronze"
table_name = "pandas_one_stock_file"
table_path = f"{lakehouse_path}/Tables/{schema_name}/{table_name}"

# only fails on missing dir
try:
    notebookutils.fs.rm(table_path, True)
except:
    pass

In [11]:
#
# 8 - write delta lake table (2 sec)
#

from deltalake import write_deltalake

# message user
print(f"create delta table located at {table_path}")

# over write 
write_deltalake(table_path, df2, mode='overwrite')


create delta table located at abfss://9d12d1b1-e32e-4bbe-9753-f0715d21ad49@onelake.dfs.fabric.microsoft.com/f880ba3e-79f9-4297-8b7f-4e39ca3d158b/Tables/bronze/pandas_one_stock_file


In [16]:
%%tsql -artifact lh_python_vs_spark -type Lakehouse

-- may take some time to refresh
-- select * from sys.objects where is_ms_shipped = 0

select top 5 * from bronze.pandas_one_stock_file where st_volume <> 0;